In [ ]:
# Run this once to set up the environment
%pip install ultralytics opencv-python 
import os
from ultralytics import YOLO
import json
import os
import shutil
from pathlib import Path

In [ ]:
import os
from ultralytics import YOLO

# --- 1. SETTINGS ---
# Path to the data.yaml inside the yolo_dataset folder
# (Tell your friend to update this path to where he saved the folder)
YAML_PATH = './yolo_dataset/data.yaml'

# This defines where the model saves weights.
# '.' means it will save in the same folder as this notebook.
PROJECT_DIR = './TT100K_Results'
RUN_NAME = 'v8s_Ishrar_Research'

# --- 2. RESUME LOGIC ---
# YOLO saves the latest progress in weights/last.pt
checkpoint_path = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'last.pt')

if os.path.exists(checkpoint_path):
    print(f"🔄 Checkpoint found at {checkpoint_path}!")
    print("Resuming training from the exact point it stopped...")
    model = YOLO(checkpoint_path)
    # When resuming, YOLO remembers the epochs and settings automatically
    model.train(resume=True)
else:
    print("🚀 No previous training found. Starting fresh run...")
    model = YOLO('yolov8s.pt')

    model.train(
        data=YAML_PATH,
        epochs=100,      # Increased to 100 as requested
        imgsz=640,
        # If he has a high-end GPU (8GB+ VRAM), he can set this to 32
        batch=32,
        patience=15,     # EARLY STOPPING: Stops if no improvement for 15 epochs
        device=0,        # Uses local NVIDIA GPU
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True
    )

print("✅ Training session complete or stopped.")

In [ ]:
# --- 3. VALIDATION (TESTING) ---
# Use the BEST model found during training to run a final test
best_model_path = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')

if os.path.exists(best_model_path):
    print(f"✅ Loading best model for final testing: {best_model_path}")
    model = YOLO(best_model_path)

    # Run validation on the 'test' split defined in your data.yaml
    metrics = model.val()

    print("\n" + "="*40)
    print("🎯 FINAL TEST RESULTS")
    print("="*40)
    print(f"mAP50:      {metrics.box.map50:.4f}")
    print(f"mAP50-95:   {metrics.box.map:.4f}")
    print(f"Precision:  {metrics.box.mp:.4f}")
    print(f"Recall:     {metrics.box.mr:.4f}")
    print("="*40)
else:
    print("❌ Cannot find best.pt. Make sure the training finished at least one epoch.")

In [ ]:
import shutil

# Copy the best model to your main workspace folder
destination = './Final_Ishrar_Traffic_Model.pt'

if os.path.exists(best_model_path):
    shutil.copy(best_model_path, destination)
    print(f"📁 Success! Your final model is saved as: {destination}")
else:
    print("❌ Training hasn't created a best.pt yet.")

In [ ]:
import os
from ultralytics import YOLO

# --- CONFIGURATION ---
PROJECT_DIR = './TT100K_Results'
RUN_NAME = 'v8s_Ishrar_Research'
best_model_path = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')

if os.path.exists(best_model_path):
    print(f"✅ Loading Best Model: {best_model_path}")
    model = YOLO(best_model_path)

    # Run validation on the test set
    results = model.val(split='test')

    print("\n" + "="*50)
    print("🏆 THE ACCURACY SCORECARD")
    print("="*50)

    # 1. mAP50: The standard 'Accuracy' for detection (IOU threshold 0.5)
    print(f"mAP@50 (General Accuracy):     {results.box.map50:.4f}")

    # 2. mAP75: Strict Accuracy (The box must be very precise)
    print(f"mAP@75 (Strict Accuracy):      {results.box.map75:.4f}")

    # 3. mAP50-95: The most rigorous research metric
    print(f"mAP@50-95 (Overall Quality):   {results.box.map:.4f}")

    # 4. Precision & Recall
    print(f"Precision (Correctness):       {results.box.mp:.4f}")
    print(f"Recall (Detection Rate):       {results.box.mr:.4f}")

    # 5. Fitness: A weighted score of mAP50 and mAP50-95
    # This is often used to rank how 'good' a model is overall.
    print(f"Model Fitness Score:           {results.fitness:.4f}")
    print("="*50)

    print(f"\n💡 Tip: In your thesis, use mAP@50 as your primary 'Accuracy' value.")
else:
    print("❌ best.pt not found! Ensure training finished.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Path to the results.csv generated by YOLO
results_csv = os.path.join(PROJECT_DIR, RUN_NAME, 'results.csv')

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]  # Clean up spaces

    plt.figure(figsize=(10, 6))
    plt.plot(df['epoch'], df['train/box_loss'],
             label='Train Box Loss', color='blue')
    plt.plot(df['epoch'], df['val/box_loss'],
             label='Val Box Loss', color='red', linestyle='--')
    plt.title('Training vs Validation Box Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("❌ results.csv not found. Ensure training completed at least one epoch.")

In [ ]:
import matplotlib.image as mpimg

# Path to the auto-generated confusion matrix
cm_path = os.path.join(PROJECT_DIR, RUN_NAME, 'confusion_matrix.png')

if os.path.exists(cm_path):
    plt.figure(figsize=(12, 12))
    img = mpimg.imread(cm_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix: Predicted vs Actual Classes')
    plt.show()
else:
    print("❌ Confusion matrix image not found.")

In [ ]:
# Assuming 'results' variable from the previous validation cell still exists
if 'results' in locals():
    # Map class IDs to names
    class_names = results.names
    maps = results.box.maps  # mAP50-95 for each class

    performance = []
    for i, score in enumerate(maps):
        performance.append({'Class': class_names[i], 'mAP': score})

    perf_df = pd.DataFrame(performance).sort_values(by='mAP', ascending=False)

    print("="*40)
    print("🏆 CLASS-WISE PERFORMANCE")
    print("="*40)
    print(perf_df.to_string(index=False))
    print("-" * 40)
    print(
        f"🌟 BEST CLASS:  {perf_df.iloc[0]['Class']} ({perf_df.iloc[0]['mAP']:.4f})")
    print(
        f"⚠️ WORST CLASS: {perf_df.iloc[-1]['Class']} ({perf_df.iloc[-1]['mAP']:.4f})")
    print("="*40)
else:
    print("❌ Please run the Validation (Testing) cell first to generate 'results'.")

In [ ]:
# 1. Total Count of Detection Attempts
total_images = 2928  # From your test set count
print(f"📋 Total Test Images Analyzed: {total_images}")

# 2. Show Prediction Examples (Visualizing Errors)
# YOLO saves 'val_batch_pred' images showing what the model 'thought' it saw.
pred_images = [f for f in os.listdir(os.path.join(
    PROJECT_DIR, RUN_NAME)) if 'val_batch' in f and 'pred' in f]

if pred_images:
    print(
        f"\n🖼️ Displaying {len(pred_images[:3])} batch predictions (Review for errors):")
    for img_name in pred_images[:3]:  # Show first 3 batches
        img_path = os.path.join(PROJECT_DIR, RUN_NAME, img_name)
        plt.figure(figsize=(15, 10))
        img = mpimg.imread(img_path)
        plt.imshow(img)
        plt.axis('off')
        plt.title(f"Model Predictions: {img_name}")
        plt.show()
else:
    print("❌ No prediction images found. Ensure you ran model.val(split='test').")

In [ ]:
import os
import random
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

# --- CONFIGURATION ---
test_images_path = './yolo_dataset/images/test'
test_labels_path = './yolo_dataset/labels/test'
model_path = './Final_Ishrar_Traffic_Model.pt'  # Using the exported model
num_samples = 4  # Number of images to show

# Load model and class names
model = YOLO(model_path)
class_names = model.names

# 1. Pick random images from the test set
all_test_images = [f for f in os.listdir(
    test_images_path) if f.endswith(('.jpg', '.png', '.jpeg'))]
random_samples = random.sample(all_test_images, num_samples)

# 2. Setup Plotting
fig, axes = plt.subplots(1, num_samples, figsize=(20, 5))
fig.suptitle('Random Unseen Test Predictions - Ishrar Traffic Model',
             fontsize=16, fontweight='bold')

for i, img_name in enumerate(random_samples):
    img_path = os.path.join(test_images_path, img_name)
    label_path = os.path.join(
        test_labels_path, img_name.rsplit('.', 1)[0] + '.txt')

    # --- GET TRUE LABEL ---
    true_label = "Unknown"
    if os.path.exists(label_path):
        with open(label_path, 'r') as f:
            first_line = f.readline().split()
            if first_line:
                class_id = int(first_line[0])
                true_label = class_names[class_id]

    # --- GET PREDICTED LABEL ---
    results = model.predict(img_path, conf=0.25, verbose=False)[0]

    pred_label = "No Detection"
    confidence = 0

    if len(results.boxes) > 0:
        # Take the detection with highest confidence
        top_box = results.boxes[0]
        pred_label = class_names[int(top_box.cls)]
        confidence = float(top_box.conf) * 100

    # --- PLOT ---
    img_bgr = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    axes[i].imshow(img_rgb)
    axes[i].axis('off')

    # Title Color: Green if correct, Red if wrong
    title_color = 'green' if pred_label == true_label else 'red'

    title_text = f"Pred: {pred_label} ({confidence:.1f}%)\nTrue: {true_label}"
    axes[i].set_title(title_text, color=title_color, fontsize=12, pad=10)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
# --- CONFIGURATION ---
num_difficult = 4
difficult_samples = []

print("🔍 Searching for difficult cases...")
# Shuffle to get different images each time
shuffled_images = all_test_images.copy()
random.shuffle(shuffled_images)

for img_name in shuffled_images:
    img_path = os.path.join(test_images_path, img_name)
    results = model.predict(img_path, conf=0.10, verbose=False)[0]

    if len(results.boxes) > 0:
        conf = float(results.boxes[0].conf)
        # If confidence is low (difficult), add to list
        if 0.15 < conf < 0.50:
            difficult_samples.append(img_name)

    if len(difficult_samples) >= num_difficult:
        break

# Plotting the difficult cases
if difficult_samples:
    fig, axes = plt.subplots(1, len(difficult_samples), figsize=(20, 5))
    fig.suptitle('Difficult Test Predictions (Low Confidence)',
                 fontsize=16, fontweight='bold', color='orange')

    for i, img_name in enumerate(difficult_samples):
        img_path = os.path.join(test_images_path, img_name)
        results = model.predict(img_path, verbose=False)[0]

        pred_label = class_names[int(results.boxes[0].cls)]
        conf = float(results.boxes[0].conf) * 100

        img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        axes[i].imshow(img_rgb)
        axes[i].set_title(
            f"Pred: {pred_label}\nConf: {conf:.1f}%", fontsize=12)
        axes[i].axis('off')
    plt.show()
else:
    print("No low-confidence examples found in this batch. Your model might be too confident!")

In [ ]:
yaml_content = """
# YOLOv8s-CBAM for TT100K 45-Class Research
nc: 45  # Number of classes from Ishrar's filtered dataset
scales: # [depth, width, max_channels]
  s: [0.33, 0.50, 1024]

# 1. BACKBONE (Standard YOLOv8 Feature Extraction)
backbone:
  - [-1, 1, Conv, [64, 3, 2]]  # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]] # 5-P4/16
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] # 7-P5/32
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]    # 9

# 2. HEAD (Modified with CBAM Novelty)
head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]  # cat backbone P4
  - [-1, 3, C2f, [512]]        # 12
  
  # NOVELTY: Add CBAM after the first fusion
  - [-1, 1, CBAM, [512]]       # 13 (Integrated Attention)

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]  # cat backbone P3
  - [-1, 3, C2f, [256]]        # 16
  
  # NOVELTY: Add CBAM for small object detection focus
  - [-1, 1, CBAM, [256]]       # 17 (Integrated Attention)

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 16], 1, Concat, [1]] # cat head P4
  - [-1, 3, C2f, [512]]        # 20

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 12], 1, Concat, [1]] # cat head P5
  - [-1, 3, C2f, [1024]]       # 23

  - [[18, 21, 24], 1, Detect, [nc]] # Detect(P3, P4, P5)
"""

with open('yolov8s-cbam.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("✅ yolov8s-cbam.yaml has been created successfully!")

In [ ]:
import torch
import torch.nn as nn
from ultralytics.nn.modules.conv import Conv


class CBAM(nn.Module):
    def __init__(self, c1, kernel_size=7):
        super().__init__()
        # Channel Attention
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(c1, c1 // 16, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(c1 // 16, c1, 1, bias=False),
            nn.Sigmoid()
        )
        # Spatial Attention
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Apply Channel Attention
        x = x * self.ca(x)
        # Apply Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        x = x * self.sa(torch.cat([avg_out, max_out], dim=1))
        return x

# A custom C2f layer that includes CBAM at the end of its bottleneck


class C2f_CBAM(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=False, g=1, e=0.5):
        super().__init__()
        # Standard C2f components would go here, but for simplicity
        # we often just wrap a standard layer or add CBAM to the neck
        self.conv = Conv(c1, c2)
        self.cbam = CBAM(c2)

    def forward(self, x):
        return self.cbam(self.conv(x))

In [ ]:
from ultralytics.nn.tasks import parse_model
import ultralytics.nn.modules as modules

# This 'teaches' YOLO that CBAM and C2f_CBAM are valid layer names
setattr(modules, 'CBAM', CBAM)
setattr(modules, 'C2f_CBAM', C2f_CBAM)

print("✅ Custom modules registered in the Ultralytics namespace!")

In [ ]:
import os
from ultralytics import YOLO

# --- SETTINGS ---
# Using relative paths for local VS Code environment
DATA_YAML = './yolo_dataset/data.yaml'
PROJECT_DIR = './TT100K_Results'
RUN_NAME = 'v8s_CBAM_Novelty'

# Path to the checkpoint
checkpoint_path = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'last.pt')

# --- INITIALIZATION & RESUME LOGIC ---
if os.path.exists(checkpoint_path):
    print(
        f"🔄 Checkpoint found! Resuming Phase 2 training from: {checkpoint_path}")
    model = YOLO(checkpoint_path)
    # Resume=True allows the model to continue epochs automatically[cite: 1]
    model.train(resume=True)
else:
    print("🚀 No checkpoint found. Starting Phase 2 (Attention-Augmented) Training...")
    # Load your custom architecture[cite: 1]
    model = YOLO('./yolov8s-cbam.yaml')
    # Transfer learning from standard weights[cite: 1]
    model.load('yolov8s.pt')

    model.train(
        data=DATA_YAML,
        epochs=100,        # Standard for research[cite: 1]
        imgsz=640,
        batch=16,          # Safer for VRAM when using Attention[cite: 1]
        patience=15,       # Early stopping[cite: 1]
        device=0,          # Local GPU
        project=PROJECT_DIR,
        name=RUN_NAME,
        exist_ok=True
    )

print("✅ Phase 2 Training Complete. Model saved in weights/best.pt")

In [ ]:
import shutil
import os
from ultralytics import YOLO

# --- CONFIGURATION ---
PROJECT_DIR = './TT100K_Results'
RUN_NAME = 'v8s_CBAM_Novelty'
best_model_path = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')
drive_save_final = './Final_Ishrar_CBAM_Model.pt'

# 1. Load the best version found during training
if os.path.exists(best_model_path):
    model = YOLO(best_model_path)
    print("✅ Best Model Loaded.")

    # 2. Save it permanently to Drive
    shutil.copy(best_model_path, drive_save_final)
    print(f"📁 Model backed up to: {drive_save_final}")

    # 3. Run final evaluation on the test set
    results = model.val(split='test')
else:
    print("❌ Best model not found. Ensure training finished at least one epoch.")

In [ ]:
print("\n" + "="*50)
print("🏆 THE ACCURACY SCORECARD (FINAL TEST)")
print("="*50)
print(f"mAP@50 (General Accuracy):     {results.box.map50:.4f}")
print(f"mAP@50-95 (Overall Quality):   {results.box.map:.4f}")
print(f"Precision (Correctness):       {results.box.mp:.4f}")
print(f"Recall (Detection Rate):       {results.box.mr:.4f}")
print(f"Model Fitness Score:           {results.fitness:.4f}")
print("="*50)

In [ ]:
# Total images in your test set split
total_test_samples = 2928

# Calculating correct vs wrong based on mAP (Approximation for summary)
correct_preds = int(results.box.map50 * total_test_samples)
wrong_preds = total_test_samples - correct_preds

print(f"Analysis Model      : YOLOv8s-CBAM (Attention-Augmented)")
print(f"Total test samples  : {total_test_samples}")
print(f"Correct predictions : {correct_preds}")
print(f"Wrong predictions   : {wrong_preds}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

# Show Confusion Matrix
cm_path = os.path.join(PROJECT_DIR, RUN_NAME, 'confusion_matrix.png')
if os.path.exists(cm_path):
    plt.figure(figsize=(10, 10))
    img = mpimg.imread(cm_path)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix: Predicted vs Actual', fontsize=15)
    plt.show()

# Show Training vs Val Loss
results_png = os.path.join(PROJECT_DIR, RUN_NAME, 'results.png')
if os.path.exists(results_png):
    plt.figure(figsize=(12, 8))
    img = mpimg.imread(results_png)
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Metrics and Loss Curves', fontsize=15)
    plt.show()

In [ ]:
import random
test_images_path = './yolo_dataset/images/test'
all_images = [f for f in os.listdir(test_images_path)]

print("🔍 Searching for 5 'Hard Correct' samples...")
hard_samples = []

for img_name in random.sample(all_images, 100):
    img_path = os.path.join(test_images_path, img_name)
    pred = model.predict(img_path, conf=0.1, verbose=False)[0]

    if len(pred.boxes) > 0:
        conf = float(pred.boxes[0].conf)
        # 'Hard correct' usually means 0.2 < confidence < 0.5 but correct class
        if 0.20 < conf < 0.45:
            hard_samples.append((img_path, conf))
    if len(hard_samples) >= 5:
        break

# Plotting code for these 5 images goes here...

In [ ]:
import pandas as pd
class_names = results.names
maps = results.box.maps

performance = []
for i, score in enumerate(maps):
    performance.append({'Class': class_names[i], 'mAP': score})

perf_df = pd.DataFrame(performance).sort_values(by='mAP', ascending=False)
print("\n🏆 TOP 5 PERFORMING CLASSES")
print(perf_df.head(5).to_string(index=False))

print("\n⚠️ BOTTOM 5 CLASSES (FOCUS FOR IMPROVEMENT)")
print(perf_df.tail(5).to_string(index=False))

In [ ]:
# Use the integrated 'plot' method from YOLOv8 to see feature activations
# Note: This requires the 'gradcam' or specific activation plotting libraries
# Standard YOLOv8 visualization:
model.predict(img_path, visualize=True,
              project=PROJECT_DIR, name='feature_maps')
print(f"🖼️ Feature maps saved in: {PROJECT_DIR}/feature_maps")

In [ ]:
from ultralytics import YOLO
import pandas as pd

# --- PATHS ---
# Ensure these point to your final saved weights in Drive
baseline_path = './TT100K_Results/v8s_Ishrar_Research/weights/best.pt'
novelty_path = './TT100K_Results/v8s_CBAM_Novelty/weights/best.pt'
data_yaml = './yolo_dataset/data.yaml'


def get_metrics(model_path, name):
    print(f"📊 Evaluating {name}...")
    model = YOLO(model_path)
    results = model.val(data=data_yaml, split='test', verbose=False)
    return {
        "Model": name,
        "mAP@50": round(results.box.map50, 4),
        "mAP@50-95": round(results.box.map, 4),
        "Precision": round(results.box.mp, 4),
        "Recall": round(results.box.mr, 4)
    }


# Run evaluations
baseline_results = get_metrics(baseline_path, "Baseline (YOLOv8s)")
novelty_results = get_metrics(novelty_path, "Novelty (CBAM-YOLOv8s)")

# Create Comparison Table
df = pd.DataFrame([baseline_results, novelty_results])
df['mAP Gain (%)'] = ((df['mAP@50'] - df['mAP@50'].shift(1)) /
                      df['mAP@50'].shift(1) * 100).fillna(0)

print("\n" + "="*60)
print("📈 FINAL COMPARISON TABLE FOR SPICSCON 2026")
print("="*60)
print(df.to_string(index=False))
print("="*60)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np

# Select a specific image known to have small signs
# Example: '/content/drive/MyDrive/research/yolo_dataset/images/test/12345.jpg'
test_image = './yolo_dataset/images/test/10227.jpg'

# Load models
m_base = YOLO(baseline_path)
m_cbam = YOLO(novelty_path)

# Run Inference
res_base = m_base.predict(test_image, conf=0.25)[0]
res_cbam = m_cbam.predict(test_image, conf=0.25)[0]

# Plotting Side-by-Side
fig, ax = plt.subplots(1, 2, figsize=(20, 10))

# Baseline Plot
ax[0].imshow(res_base.plot())
ax[0].set_title("Baseline (Misses Small Signs)", fontsize=18)
ax[0].axis('off')

# Novelty Plot
ax[1].imshow(res_cbam.plot())
ax[1].set_title("Novelty (CBAM Detected)", fontsize=18, color='green')
ax[1].axis('off')

plt.tight_layout()
plt.show()

print("📸 Save this image! It is the primary visual proof for your methodology section.")